In [1]:
#https://huggingface.co/mistralai/Mistral-7B-Instruct-v0.3
# ===============================
# 1. Instalacje
# ===============================
!pip install -U bitsandbytes sentence-transformers faiss-cpu transformers accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 60.2 MB/s eta 0:00:00


In [2]:
# ===============================
# 2. Importy
# ===============================
import torch
import numpy as np
import faiss
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    pipeline,
    BitsAndBytesConfig
)
from sentence_transformers import SentenceTransformer

In [3]:
# ===============================
# 3. Model językowy – Mistral
# ===============================
# Cek: Zamienic Bielik na inny model (Mistral) zachowujac ta sama architekture RAG:
# Architektura: embeddings -> FAISS -> kontekst -> promt -> generator

# Wybrany model
model_name = "mistralai/Mistral-7B-Instruct-v0.3"


# Mistral nie ma domyslnego pad_token; ustawiamy go jawnie na eos_token
# zeby pipeline nie wypisywal komunikatu "Setting pad_token_id..." przy kazdej generacji.
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token


# 4-bit quantization (bitsandbytes) -> mniejsze zuzycie VRAM, latwiejsze uruchomienie w Colab na GPU
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

# Wymuszenie zaladowania modelu w calosci na GPU:0.
# device_map="auto" potrafil przerzucac czesc warstw na CPU/dysk i powodowac bledy / dlugie ladowanie.
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map={"": 0},
    quantization_config=bnb_config
)

# Pipeline do generacji tekstu; ten obiekt bedzie uzywany w ask_bot() do generowania odpowiedzi.
generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer
)

# Flaga pomocnicza: informacja, ze prompt w ask_bot() owijamy w format Mistrala: [INST] ... [/INST]
PROMPT_STYLE = "mistral_inst"

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

...

model-00002-of-00003.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

model-00003-of-00003.safetensors:   0%|          | 0.00/4.55G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

Device set to use cuda:0


In [24]:
# ===============================
# 4. Dokument źródłowy (REGULAMIN)
# ===============================
document = """
Procedura reagowania na incydenty bezpieczeństwa IT w organizacji

1. Cel procedury
Celem procedury jest zapewnienie spójnego i skutecznego reagowania na incydenty bezpieczeństwa IT w organizacji oraz minimalizacja ich skutków.

2. Definicja incydentu
Incydentem bezpieczeństwa IT jest każde zdarzenie, które może prowadzić do naruszenia poufności, integralności lub dostępności systemów informatycznych lub danych.

3. Przykłady incydentów
Do incydentów zalicza się między innymi: nieautoryzowany dostęp do systemu, wyciek danych, infekcję złośliwym oprogramowaniem, ataki phishingowe, ataki typu DDoS oraz utratę urządzenia służbowego.

4. Role i odpowiedzialności
Użytkownik końcowy jest zobowiązany do niezwłocznego zgłoszenia podejrzanego zdarzenia.
Administrator IT odpowiada za analizę techniczną incydentu.
Kierownik zespołu IT odpowiada za decyzję o eskalacji incydentu.
Zespół bezpieczeństwa IT odpowiada za koordynację działań naprawczych.

5. Zgłaszanie incydentu
Incydent należy zgłosić poprzez dedykowany system zgłoszeń IT lub drogą mailową.
Zgłoszenie powinno zawierać opis zdarzenia, czas wystąpienia oraz systemy, których dotyczy.

6. Klasyfikacja incydentu
Po otrzymaniu zgłoszenia administrator IT klasyfikuje incydent jako niski, średni lub krytyczny.
Klasyfikacja zależy od skali wpływu na systemy oraz potencjalnych skutków dla organizacji.

7. Reakcja na incydent
W przypadku incydentu niskiego poziomu administrator IT podejmuje działania naprawcze samodzielnie.
W przypadku incydentu średniego lub krytycznego kierownik zespołu IT podejmuje decyzję o eskalacji.

8. Eskalacja
Incydenty krytyczne są eskalowane do zespołu bezpieczeństwa IT.
W przypadku incydentów o charakterze prawnym informowany jest dział prawny organizacji.

9. Działania naprawcze
Działania naprawcze obejmują izolację zagrożonych systemów, usunięcie przyczyny incydentu oraz przywrócenie poprawnego działania usług.

10. Przywracanie systemów
Systemy przywracane są na podstawie aktualnych kopii zapasowych.
Po przywróceniu systemów wykonywane są testy poprawności działania.

11. Dokumentacja incydentu
Każdy incydent musi zostać udokumentowany w systemie zgłoszeń IT.
Dokumentacja zawiera opis incydentu, podjęte działania oraz wnioski.

12. Zamknięcie incydentu
Incydent uznaje się za zamknięty po usunięciu skutków oraz zatwierdzeniu przez kierownika zespołu IT.

13. Przegląd po incydencie
Po incydencie krytycznym przeprowadzany jest przegląd w celu zapobiegania podobnym zdarzeniom w przyszłości.

14. Zakres procedury
Procedura dotyczy wyłącznie systemów informatycznych organizacji.
Procedura nie obejmuje incydentów niezwiązanych z infrastrukturą IT ani zdarzeń losowych niezależnych od systemów informatycznych.

15. Postanowienia końcowe
Nieprzestrzeganie procedury może skutkować konsekwencjami służbowymi zgodnie z obowiązującymi zasadami organizacji.

"""

In [39]:
# ===============================
# 5. Chunking – każda linijka to osobny chunk
# ===============================
import re

def chunk_text(text):
    chunks = []
    pending_header = None

    for raw in text.split("\n"):
        line = raw.strip()
        if not line:
            continue

        is_header = bool(re.match(r"^\d+\.\s+.+$", line)) and len(line.split()) <= 6

        if is_header:
            pending_header = line
            continue

        if pending_header:
            chunks.append(pending_header + " " + line)
            pending_header = None
        else:
            chunks.append(line)

    # jeśli dokument kończy się nagłówkiem bez treści
    if pending_header:
        chunks.append(pending_header)

    return chunks


In [40]:
# ===============================
# 6. Embeddingi (polski + multi)
# ===============================
# Cel: zamienic tekst (pytania i chunki dokumentu) na wektory, ktore da sie porownywac.
# normalize_embeddings=True + IndexFlatIP => score ~ cosine similarity (im wyzszy, tym lepsze dopasowanie).
embedder = SentenceTransformer("intfloat/multilingual-e5-base")

chunk_embeddings = embedder.encode(
    chunks,
    normalize_embeddings=True
)

dim = chunk_embeddings.shape[1]
index = faiss.IndexFlatIP(dim)  # inner product na znormalizowanych wektorach = cosine similarity
index.add(np.array(chunk_embeddings))

In [49]:
# ===============================
# 7. Retrieval (FAISS)
# ===============================
# Cel: wyszukac najbardziej podobne fragmenty dokumentu (chunks) na podstawie embeddingow.
# Funkcja wspiera punkt 5 projektu:
# - k (ile fragmentow zwracamy),
# - score_threshold (minimalny prog podobienstwa; ponizej odrzucamy),
# - meta (do analizy: jakie wyniki byly i jakie zostaly po odfiltrowaniu).
def retrieve_context(query, k=3, score_threshold=None, return_meta=False):
    # embedding zapytania (normalizowany)
    q_emb = embedder.encode([query], normalize_embeddings=True)  # shape: (1, dim)

    # FAISS: IndexFlatIP -> zwraca inner product (dla znormalizowanych wektorow ~ cos similarity)
    scores, indices = index.search(np.array(q_emb), k)  # scores shape: (1,k), indices shape: (1,k)

    results_all = []
    results_kept = []

    for rank, (score, idx) in enumerate(zip(scores[0], indices[0]), start=1):
        if idx == -1:
            continue

        item = {
            "rank": rank,
            "idx": int(idx),
            "score": float(score),
            "text": chunks[int(idx)]
        }
        results_all.append(item)

        # filtr progowy (jesli ustawiony)
        if (score_threshold is None) or (score >= score_threshold):
            results_kept.append(item)

    context = "\n\n".join([r["text"] for r in results_kept])

    if not return_meta:
        return context

    meta = {
        "k": k,
        "score_threshold": score_threshold,
        "results_all": results_all,
        "results_kept": results_kept
    }
    return context, meta


In [50]:
# ===============================
# 8. Funkcja ask_bot
# ===============================
# Cel: polaczyc retrieval (FAISS) + generacje (LLM) w jeden przeplyw:
# pytanie -> kontekst z dokumentu -> prompt -> krotka odpowiedz.
# Funkcja wspiera punkt 5 projektu: temperature, k, score_threshold.
def ask_bot(question, k=3, score_threshold=None, temperature=0.01, show_meta=False):
    context, meta = retrieve_context(
        question,
        k=k,
        score_threshold=score_threshold,
        return_meta=True
    )

    print("-" * 80)
    print("PYTANIE:")
    print(question)
    print("-" * 40)

    # META (dla obrony: pokazuje jak dziala FAISS i prog)
    if show_meta:
        print("META (FAISS):")
        print(f"- k={meta['k']}, score_threshold={meta['score_threshold']}")
        print(f"- results_all={len(meta['results_all'])}, results_kept={len(meta['results_kept'])}")
        # pokaz top wyniki (max 5) zeby bylo czytelnie
        for r in meta["results_all"][:5]:
            kept_flag = "KEPT" if r in meta["results_kept"] else "DROP"
            preview = r["text"].replace("\n", " ")
            preview = (preview[:90] + "...") if len(preview) > 90 else preview
            print(f"  #{r['rank']} score={r['score']:.4f} {kept_flag} | {preview}")
        print("-" * 40)

    # KONTEKST
    print("KONTEKST:")
    if context.strip():
        print(context)
    else:
        print("Brak pasujacych fragmentow.")
    print("-" * 40)

    # Jesli nie ma kontekstu -> twarda odmowa (spelnia punkt 4 i 3)
    if not context.strip():
        print("ODPOWIEDŹ:")
        print("Brak informacji w dokumencie.")
        print("-" * 80)
        return

    base_prompt = f"""
Jestes asystentem, ktory odpowiada WYLACZNIE na podstawie DOKUMENTU.
NIE WOLNO dodawac wiedzy spoza KONTEKSTU ani wyciagac wnioskow (inferencji).

Zasady:
1) Odpowiadasz tylko na podstawie tresci z sekcji KONTEKST.
2) Jesli w KONTEKST nie ma informacji potrzebnej do odpowiedzi, zwroc DOKLADNIE:
Brak informacji w dokumencie.
3) Odpowiedz ma byc krotka: maks. 1 zdanie, jedna linia, bez wypunktowan.
4) Jesli pytanie prosi o "przyklady" lub "wymien", a KONTEKST zawiera liste, wypisz elementy listy po przecinkach w jednym zdaniu.

KONTEKST:
{context}

PYTANIE:
{question}

ODPOWIEDŹ: (krotko, jedna linia)"""

    # Mistral Instruct wrapper
    prompt = f"[INST] {base_prompt} [/INST]"

    output = generator(
        prompt,
        max_new_tokens=70,
        temperature=temperature,
        do_sample=(temperature > 0),
        return_full_text=False
    )[0]["generated_text"]

    answer = output.split("ODPOWIEDŹ:")[-1].strip()
    answer = answer.split("\n")[0].strip()

    print("ODPOWIEDŹ:")
    print(answer)
    print("-" * 80)


In [53]:
# ===============================
# 9. Testy
# ===============================
BASE_TH = 0.35

# 1) RAG działa
ask_bot("Jaki jest cel procedury reagowania na incydenty?", k=3, score_threshold=BASE_TH, temperature=0.01)
ask_bot("Jak należy zgłosić incydent według dokumentu?", k=3, score_threshold=BASE_TH, temperature=0.01)

# 2) Odmowa spoza dokumentu
ask_bot("W jakiej temperaturze wrze woda?", k=3, score_threshold=BASE_TH, temperature=0.01)

# 3) score_threshold
ask_bot("Jak należy zgłosić incydent?", k=3, score_threshold=0.20, temperature=0.01, show_meta=True)
ask_bot("Jak należy zgłosić incydent?", k=3, score_threshold=0.89, temperature=0.01, show_meta=True)

# 4) k
ask_bot("Kto odpowiada za analizę techniczną i kto za eskalację incydentu?", k=1, score_threshold=BASE_TH, temperature=0.01, show_meta=True)
ask_bot("Kto odpowiada za analizę techniczną i kto za eskalację incydentu?", k=5, score_threshold=BASE_TH, temperature=0.01, show_meta=True)

# 5) temperature
Q_LISTA = "Wymien przyklady incydentow bezpieczenstwa IT podane w dokumencie."

ask_bot(Q_LISTA, k=5, score_threshold=0.20, temperature=0.01, show_meta=True)
ask_bot(Q_LISTA, k=5, score_threshold=0.20, temperature=0.50, show_meta=True)
ask_bot(Q_LISTA, k=5, score_threshold=0.20, temperature=0.90, show_meta=True)



--------------------------------------------------------------------------------
PYTANIE:
Jaki jest cel procedury reagowania na incydenty?
----------------------------------------
KONTEKST:
Procedura reagowania na incydenty bezpieczeństwa IT w organizacji

Celem procedury jest zapewnienie spójnego i skutecznego reagowania na incydenty bezpieczeństwa IT w organizacji oraz minimalizacja ich skutków.

7. Reakcja na incydent
----------------------------------------
ODPOWIEDŹ:
Zapewnienie spójnego i skutecznego reagowania na incydenty bezpieczeństwa IT w organizacji oraz minimalizacja ich skutków.
--------------------------------------------------------------------------------
--------------------------------------------------------------------------------
PYTANIE:
Jak należy zgłosić incydent według dokumentu?
----------------------------------------
KONTEKST:
Incydent należy zgłosić poprzez dedykowany system zgłoszeń IT lub drogą mailową.

Każdy incydent musi zostać udokumentowany w system